In [5]:
%load_ext autoreload
%autoreload 2

In [6]:

from request_seges import Seges as sg
from request_seges import Login
from urllib.parse import urlparse
import urllib3

import requests

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

session = requests.Session()
usuario = '10631094776'
senha = usuario

login = Login("https://seges.sedu.es.gov.br")

session_logada, base_url = login.autenticar(usuario, senha)


In [7]:
etapa = 0 # significa que é o trimestre (0-1ºTrimestre, 1-2ºTrimestre...)
seges = sg(session_logada, etapa, base_url)

In [8]:
# pega links das turmas
url = 'https://seges.sedu.es.gov.br/avaliacao_modo_avancados/turmas'
lancar_notas = seges.get_minhas_turmas(url)

In [9]:
# pega minhas notas
listagem_avaliacao = seges.get_links_minhas_notas(lancar_notas['href'])
listagem_avaliacao.tail()
'''
output
classroom é a turma
discipline_id é o tipo de diciplina
stage_id é o código do trimestre
'''

'\noutput\nclassroom é a turma\ndiscipline_id é o tipo de diciplina\nstage_id é o código do trimestre\n'

In [10]:
minhas_turmas = (listagem_avaliacao.drop_duplicates(subset='classroom_id').reset_index(drop=True))
meus_alunos = seges.get_alunos_por_turma(minhas_turmas['href'].to_list())
minhas_avaliacoes = seges.get_avaliacoes(listagem_avaliacao['href'])
listagem_avaliacao["classroom_id"] = listagem_avaliacao["classroom_id"].astype(int)

notas = seges.get_notas(listagem_avaliacao['href'])

In [11]:
turma = '1ªIM01-EM-LCH'
numero_aluno = 1
atividade_nome = 'LISTAS DE EXERCICIOS'
disciplina = 'QUÍMICA'

aluno = meus_alunos[(meus_alunos['numero'] == numero_aluno) & (meus_alunos['turma'] == turma)]
classroom_id = aluno.iloc[0]['classroom_id']
disciplina = listagem_avaliacao[(listagem_avaliacao['disciplina'] == disciplina) & (listagem_avaliacao['classroom_id'] == classroom_id)]
avaliacao = minhas_avaliacoes[(minhas_avaliacoes['turma'] == turma) & (minhas_avaliacoes['atividade_nome'] == atividade_nome)]

In [12]:

turma = '1ªIM01-EM-LCH'
numero_aluno = 2
atividade_nome = 'LISTAS DE EXERCICIOS'
disciplina_nome = 'QUÍMICA'

recovery = 2

payload = seges.montar_payload_nota(meus_alunos, listagem_avaliacao, minhas_avaliacoes, notas, 
                                    numero_aluno, turma, disciplina_nome, atividade_nome)

payload['recovery'] = recovery


In [ ]:
'''
Envio de notas
seges.alterar_nota(payload)
'''

In [ ]:
'''
envio para git
'''
